In [1]:
!pip -q install datasets transformers sentence-transformers faiss-cpu rank-bm25 evaluate accelerate scikit-learn gensim nltk gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import re
import gc
import json
import time
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

from datasets import load_dataset, Dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    pipeline
)

from sentence_transformers import SentenceTransformer, CrossEncoder

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import gradio as gr

from gensim.models import Word2Vec, FastText

warnings.filterwarnings('ignore')

In [3]:
for pkg in ["punkt", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except:
        pass

## 1. Load Dataset

In [4]:
dataset = load_dataset(
    "PolyAI/banking77",
    revision="830c4f2be6949546b11fe83fbc50993348a2bccd"
)
dataset

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/295k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path("./finassist_project")
PROJECT_DIR.mkdir(exist_ok=True, parents=True)
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True, parents=True)
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(exist_ok=True, parents=True)

In [6]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

label_names = dataset["train"].features["label"].names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}

train_df["label_name"] = train_df["label"].map(id2label)
test_df["label_name"] = test_df["label"].map(id2label)

In [7]:
print(train_df.shape, test_df.shape)
train_df.head()

(10003, 3) (3080, 3)


,text,label,label_name
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival
3,Can I track my card while it is in the process...,11,card_arrival
4,"How do I know if I will get my card, or if it ...",11,card_arrival


In [8]:
train_df["text_len_words"] = train_df["text"].str.split().str.len()
train_df["text_len_chars"] = train_df["text"].str.len()

display(train_df[["text_len_words", "text_len_chars"]].describe())
display(train_df["label_name"].value_counts().head(15))

,text_len_words,text_len_chars
count,10003.000000,10003.000000
mean,11.949415,59.473758
std,7.891577,40.867901
min,2.000000,13.000000
25%,7.000000,36.000000
50%,10.000000,47.000000
75%,13.000000,64.000000
max,79.000000,433.000000


,count
label_name,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
transaction_charged_twice,175
declined_cash_withdrawal,173
transfer_fee_charged,172
balance_not_updated_after_bank_transfer,171


## 2. Text preprocessing


In [9]:
STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
def basic_clean(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " <MENTION> ", text)
    text = re.sub(r"#[A-Za-z0-9_]+", " <HASHTAG> ", text)
    text = re.sub(r"\d+", " <NUMBER> ", text)
    text = re.sub(r"[^a-z<>\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def classical_preprocess(text : str) ->str:
    text = basic_clean(text)
    tokens = []
    for tok in text.split():
        if tok in STOPWORDS:
            continue
        if tok.startswith("<") and tok.endswith(">"):
            tokens.append(tok)
        else:
            tokens.append(lemmatizer.lemmatize(tok))
    return " ".join(tokens)

In [11]:
train_df["text_clean"] = train_df["text"].apply(classical_preprocess)
test_df["text_clean"] = test_df["text"].apply(classical_preprocess)

train_df[["text", "text_clean"]].head(10)

,text,text_clean
0,I am still waiting on my card?,still waiting card
1,What can I do if my card still hasn't arrived ...,card still arrived < > week
2,I have been waiting over a week. Is the card s...,waiting week card still coming
3,Can I track my card while it is in the process...,track card process delivery
4,"How do I know if I will get my card, or if it ...",know get card lost
5,When did you send me my new card?,send new card
6,Do you have info about the card on delivery?,info card delivery
7,What do I do if I still have not received my n...,still received new card
8,Does the package with my card have tracking?,package card tracking
9,I ordered my card but it still isn't here,ordered card still


## 3. Classical baseline: TF-IDF + Logistic Regression


In [12]:
tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=50000
    )),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

tfidf_clf.fit(train_df["text_clean"], train_df["label_name"])
tfidf_preds = tfidf_clf.predict(test_df["text_clean"])

tfidf_acc = accuracy_score(test_df["label_name"], tfidf_preds)
tfidf_f1 = f1_score(test_df["label_name"], tfidf_preds, average="weighted")

print("TF-IDF Accuracy:", round(tfidf_acc, 4))
print("TF-IDF Weighted F1:", round(tfidf_f1, 4))

TF-IDF Accuracy: 0.8523
TF-IDF Weighted F1: 0.8526


## 4. Dense embeddings: Word2Vec and FastText

In [13]:
tokenized_sentences = [x.split() for x in train_df["text_clean"].tolist()]

w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=SEED
)

ft_model = FastText(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=SEED
)

print("FastText OOV-style vector:", ft_model.wv["virtualcard"][:5])

FastText OOV-style vector: [-0.1952327  -0.19901037  0.20326313  0.00211654 -0.00340311]


In [14]:
def average_embedding(text, model, dim=100):
    tokens = text.split()
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)
    
    return np.mean(vecs, axis=0).astype(np.float32)

x_train_w2v = np.stack([average_embedding(t, w2v_model, 100) for t in train_df["text_clean"]])
x_test_w2v = np.stack([average_embedding(t, w2v_model, 100) for t in test_df["text_clean"]])

In [15]:
w2v_lr = LogisticRegression(max_iter=2000, class_weight='balanced')
w2v_lr.fit(x_train_w2v, train_df["label_name"])

w2v_preds = w2v_lr.predict(x_test_w2v)

w2v_acc = accuracy_score(test_df["label_name"], w2v_preds)
w2v_f1 = f1_score(test_df["label_name"], w2v_preds, average='weighted')

print("Word2Vec AvgEmb Accuracy:", round(w2v_acc, 4))
print("Word2Vec AvgEmb Weighted F1:", round(w2v_f1, 4))

Word2Vec AvgEmb Accuracy: 0.5831
Word2Vec AvgEmb Weighted F1: 0.5722


## 5. BiLSTM baseline

In [16]:
MAX_VOCAB = 20000
MAX_LEN = 40
EMBED_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 128
EPOCHS_LSTM = 8
DEVICE = "cuda"

In [17]:
label_encoder = LabelEncoder()
word_counter = Counter()

y_train_enc = label_encoder.fit_transform(train_df["label_name"])
y_test_enc = label_encoder.transform(test_df["label_name"])

In [18]:
for text in train_df["text_clean"]:
    word_counter.update(text.split())
    
special_tokens = ["<PAD>", "<UNK>"]
vocab_words = [w for w, _ in word_counter.most_common(MAX_VOCAB - len(special_tokens))]
vocab = {w: i for i, w in enumerate(special_tokens + vocab_words)}

pad_id = vocab["<PAD>"]
unk_id = vocab["<UNK>"]

In [19]:
def encode_text(text, max_len=MAX_LEN):
    ids = [vocab.get(tok, unk_id) for tok in text.split()][:max_len]
    if len(ids) < max_len:
        ids += [pad_id] * (max_len - len(ids))
    return ids

x_train_ids = np.array([encode_text(t) for t in train_df["text_clean"]], dtype=np.int64)
x_test_ids = np.array([encode_text(t) for t in test_df["text_clean"]], dtype=np.int64)

In [20]:
from torch.utils.data import Dataset as TorchDataset, DataLoader

class TextDataset(TorchDataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    
train_loader = DataLoader(TextDataset(x_train_ids, y_train_enc), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TextDataset(x_test_ids, y_test_enc), batch_size=BATCH_SIZE, shuffle=False)

In [21]:
embedding_matrix = np.random.normal(0, 0.1, size=(len(vocab), EMBED_DIM)).astype(np.float32)
embedding_matrix[pad_id] = 0

for word, idx in vocab.items():
    if word in ft_model.wv:
        embedding_matrix[idx] = ft_model.wv[word]

In [22]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, embedding_matrix=None):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        h = self.dropout(h)
        return self.fc(h)

In [23]:
criterion = nn.CrossEntropyLoss()

lstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(label_encoder.classes_),
    embedding_matrix=embedding_matrix
).to(DEVICE)

optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=2e-4)

In [24]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses, preds_all, true_all = [], [], []
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        preds_all.extend(logits.argmax(dim=1).detach().cpu().numpy())
        true_all.extend(yb.detach().cpu().numpy())
    return np.mean(losses), accuracy_score(true_all, preds_all), f1_score(true_all, preds_all, average="weighted")

In [25]:
@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    losses, preds_all, true_all = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        losses.append(loss.item())
        preds_all.extend(logits.argmax(dim=1).cpu().numpy())
        true_all.extend(yb.cpu().numpy())
    return np.mean(losses), accuracy_score(true_all, preds_all), f1_score(true_all, preds_all, average="weighted"), preds_all

In [26]:
for epoch in range(EPOCHS_LSTM):
    tr_loss, tr_acc, tr_f1 = train_one_epoch(lstm_model, train_loader, optimizer, criterion, DEVICE)
    va_loss, va_acc, va_f1, _ = eval_model(lstm_model, test_loader, criterion, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS_LSTM} | train_acc={tr_acc:.4f} | val_acc={va_acc:.4f} | val_f1={va_f1:.4f}")

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 1/8 | train_acc=0.0159 | val_acc=0.0140 | val_f1=0.0011


  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 2/8 | train_acc=0.0192 | val_acc=0.0224 | val_f1=0.0017


  0%|          | 0/79 [00:00<?, ?it/s]

...

Epoch 6/8 | train_acc=0.0752 | val_acc=0.0825 | val_f1=0.0473


  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 7/8 | train_acc=0.1015 | val_acc=0.1201 | val_f1=0.0795


  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 8/8 | train_acc=0.1362 | val_acc=0.1487 | val_f1=0.1033


In [27]:
lstm_loss, lstm_acc, lstm_f1, lstm_pred_ids = eval_model(lstm_model, test_loader, criterion, DEVICE)
lstm_preds = label_encoder.inverse_transform(np.array(lstm_pred_ids))
print("BiLSTM Accuracy:", round(lstm_acc, 4))
print("BiLSTM Weighted F1:", round(lstm_f1, 4))

BiLSTM Accuracy: 0.1487
BiLSTM Weighted F1: 0.1033


## 6. Transformer fine-tuning with DistilBERT

In [29]:
train_hf = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
test_hf = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

In [30]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenized_batch(batch):
    return tokenizer(batch["text"], ttruncation=True, padding="max_length", max_length=64)

train_tok = train_hf.map(tokenized_batch, batched=True)
test_tok = test_hf.map(tokenized_batch, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [31]:
train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

In [32]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy" : metric_acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1_weighted": metric_f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }


transformer_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "distilbert_banking77"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,2.160738,1.946862,0.690584,0.652446
2,0.927651,0.866914,0.848377,0.840420
3,0.529608,0.545189,0.892857,0.891766
4,0.363556,0.426586,0.909091,0.909014
5,0.274362,0.396519,0.913636,0.913522


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3130, training_loss=1.1565568719809047, metrics={'train_runtime': 236.1887, 'train_samples_per_second': 211.759, 'train_steps_per_second': 13.252, 'total_flos': 835159268108832.0, 'train_loss': 1.1565568719809047, 'epoch': 5.0})

In [36]:
transformer_metrics = trainer.evaluate()
transformer_preds_obj = trainer.predict(test_tok)

transformer_pred_ids = np.argmax(transformer_preds_obj.predictions, axis=1)
transformer_pred_labels = [id2label[i] for i in transformer_pred_ids]

In [37]:
transformer_acc = accuracy_score(test_df["label_name"], transformer_pred_labels)
transformer_f1 = f1_score(test_df["label_name"], transformer_pred_labels, average="weighted")


comparison = pd.DataFrame([
    {"model": "TF-IDF + LogisticRegression", "accuracy": tfidf_acc, "f1_weighted": tfidf_f1},
    {"model": "Word2Vec Avg + LogisticRegression", "accuracy": w2v_acc, "f1_weighted": w2v_f1},
    {"model": "BiLSTM + FastText init", "accuracy": lstm_acc, "f1_weighted": lstm_f1},
    {"model": "DistilBERT fine-tuned", "accuracy": transformer_acc, "f1_weighted": transformer_f1},
]).sort_values("f1_weighted", ascending=False)

comparison

,model,accuracy,f1_weighted
3,DistilBERT fine-tuned,0.913636,0.913522
0,TF-IDF + LogisticRegression,0.852273,0.852648
1,Word2Vec Avg + LogisticRegression,0.583117,0.572190
2,BiLSTM + FastText init,0.148701,0.103320


## 7. Error analysis

In [38]:
error_df = test_df.copy()
error_df["tfidf_pred"] = tfidf_preds
error_df["lstm_pred"] = lstm_preds
error_df["transformer_pred"] = transformer_pred_labels

transformer_errors = error_df[error_df["label_name"] != error_df["transformer_pred"]].copy()
transformer_errors[["text", "label_name", "transformer_pred"]].head(10)

,text,label_name,transformer_pred
0,How do I locate my card?,card_arrival,card_linking
3,Is there a way to know when my card will arrive?,card_arrival,card_delivery_estimate
5,When will I get my card?,card_arrival,card_delivery_estimate
11,How long does a card delivery take?,card_arrival,card_delivery_estimate
32,How do I know when my card will arrive?,card_arrival,card_delivery_estimate
138,Why am I being charged more ?,card_payment_wrong_exchange_rate,card_payment_fee_charged
150,the conversion value for my card payments is i...,card_payment_wrong_exchange_rate,cancel_transfer
156,How can I check the exchange rate applied to m...,card_payment_wrong_exchange_rate,exchange_rate
173,Where did this fee come from?,extra_charge_on_statement,card_payment_fee_charged
232,"I tried to take money from my card, but it did...",pending_cash_withdrawal,transfer_not_received_by_recipient


## 8. Knowledge base for retrieval and grounded answering

In [40]:
knowledge_docs = [
    {"doc_id": "kb_001", "title": "Card Arrived Damaged", "topic": "cards", "intent_hint": "card_damage", "content": "If a card arrives damaged, the customer should not use it. A replacement request can be raised after identity verification. The damaged card is deactivated and a new card is shipped."},
    {"doc_id": "kb_002", "title": "Cash Withdrawal Charge Policy", "topic": "fees", "intent_hint": "cash_withdrawal_charge", "content": "Cash withdrawal fees depend on ATM network, country, and plan tier. The app may show the final fee only after settlement."},
    {"doc_id": "kb_003", "title": "Beneficiary Not Allowed", "topic": "bank_transfer", "intent_hint": "beneficiary_not_allowed", "content": "A beneficiary may be blocked because of invalid details, recipient-bank restrictions, unsupported destination rules, or compliance checks."},
    {"doc_id": "kb_004", "title": "Pending Bank Transfer", "topic": "bank_transfer", "intent_hint": "outgoing_transfer_pending", "content": "Pending transfers may be under fraud checks, compliance review, or bank-network confirmation. Escalate only after the normal SLA has passed."},
    {"doc_id": "kb_005", "title": "Cash Withdrawal Declined", "topic": "cards", "intent_hint": "cash_withdrawal", "content": "Cash withdrawal failures can happen because of insufficient balance, ATM-side issues, chip issues, regional restrictions, or disabled cash-withdrawal controls."},
    {"doc_id": "kb_006", "title": "Transfer Not Received", "topic": "bank_transfer", "intent_hint": "beneficiary_not_allowed, transfer_not_received", "content": "If a transfer was marked completed but not received, verify the transfer reference, destination details, and business-day timeline before escalation."},
    {"doc_id": "kb_007", "title": "Card Not Received", "topic": "cards", "intent_hint": "card_not_received", "content": "Check shipping estimate, delivery address, and courier tracking. If the delivery window has passed, start a replacement workflow after verification."},
    {"doc_id": "kb_008", "title": "Cash Deposit Missing", "topic": "cash_deposit", "intent_hint": "cash_deposit", "content": "Cash deposits can take time to settle. Ask for deposit time, reference number, and ATM location. Escalate with receipt evidence if settlement exceeds the expected time."},
    {"doc_id": "kb_009", "title": "Card Security Controls", "topic": "security", "intent_hint": "cash_withdrawal, card_payment_fee_charged", "content": "Customers can control online payments, magnetic stripe usage, regional usage, and cash withdrawals from the app. Many card issues are solved by reviewing these controls."},
    {"doc_id": "kb_010", "title": "Cash Withdrawal Dispute", "topic": "disputes", "intent_hint": "cash_withdrawal_charge", "content": "A dispute may be raised when cash was not dispensed correctly or charges appear wrong after settlement. Ask for ATM location, date, amount, and receipt if available."}
]
kb_extra_docs = [
    {"doc_id": "kb_011", "title": "Bank Transfer Failed", "topic": "bank_transfer", "intent_hint": "beneficiary_not_allowed, transfer_not_received, outgoing_transfer_pending", "content": "A bank transfer can fail because of invalid recipient details, temporary bank-network issues, fraud screening, compliance review, or unsupported destination rules. Verify the account details and retry only if the error is not related to compliance restrictions."},

    {"doc_id": "kb_012", "title": "Bank Transfer Fee Charged", "topic": "fees", "intent_hint": "transfer_fee_charged", "content": "Transfer fees may depend on destination bank, transfer method, currency conversion, and the customer plan. Some charges appear only after settlement. Always check whether the transfer was domestic or international before escalating a fee complaint."},

    {"doc_id": "kb_013", "title": "Card Payment Reversed", "topic": "card_payments", "intent_hint": "cash_withdrawal_charge, card_payment_fee_charged", "content": "A reversed card payment means the original transaction was cancelled or not fully captured by the merchant. The temporary deduction may disappear automatically after settlement. If the reversal remains inconsistent after the expected timeline, raise a payment investigation."},

    {"doc_id": "kb_014", "title": "Cash Withdrawal Card Retained by ATM", "topic": "cards", "intent_hint": "cash_withdrawal", "content": "If an ATM keeps the card, advise the customer not to retry immediately. Ask for ATM location, bank name, date, and time. Some ATMs require the customer to contact the ATM owner directly, while others require card freezing and replacement."},

    {"doc_id": "kb_015", "title": "Card Contactless Not Working", "topic": "cards", "intent_hint": "card_damage, card_payment_fee_charged", "content": "Contactless card payments may fail because of damaged NFC chip, disabled contactless settings, merchant terminal issues, or repeated low-value transaction limits. Ask the customer to test chip-and-PIN and review card controls before requesting replacement."},

    {"doc_id": "kb_016", "title": "Card Frozen or Temporarily Locked", "topic": "security", "intent_hint": "cash_withdrawal, card_not_received", "content": "A card may be frozen because of suspicious activity, customer action in the app, or security review. Before escalating a cash withdrawal or payment issue, verify whether the card is currently frozen, locked, or restricted in the security settings."},

    {"doc_id": "kb_017", "title": "Cash Withdrawal Limit Reached", "topic": "cards", "intent_hint": "cash_withdrawal", "content": "Cash withdrawal may fail if the daily or monthly ATM limit has been reached. Customers should review withdrawal limits in the app, check recent ATM usage, and confirm whether regional ATM restrictions apply."},

    {"doc_id": "kb_018", "title": "Cash Deposit ATM Error", "topic": "cash_deposit", "intent_hint": "cash_deposit", "content": "Cash deposit problems may happen due to ATM hardware issues, note validation failure, partial acceptance of notes, or temporary settlement delay. Always ask for ATM location, deposit amount, time, and receipt details."},

    {"doc_id": "kb_019", "title": "Cash Deposit Rejected Notes", "topic": "cash_deposit", "intent_hint": "cash_deposit", "content": "An ATM may reject notes because they are folded, damaged, suspicious, or unsupported by the machine. If the notes were not accepted, confirm whether the customer still has them and whether the ATM completed any partial deposit."},

    {"doc_id": "kb_020", "title": "Card Delivery Delayed", "topic": "cards", "intent_hint": "card_not_received", "content": "Card delivery delays may happen because of courier backlog, address mismatch, customs checks, or incomplete verification. Check the expected delivery window, tracking details, and whether the card was returned to sender."},

    {"doc_id": "kb_021", "title": "Replacement Card Process", "topic": "cards", "intent_hint": "card_damage, card_not_received", "content": "Replacement cards are usually issued after identity verification and confirmation of the delivery address. The old card may be deactivated before the new one is shipped, depending on the reason for replacement."},

    {"doc_id": "kb_022", "title": "Recipient Bank Delay", "topic": "bank_transfer", "intent_hint": "outgoing_transfer_pending, transfer_not_received", "content": "A completed transfer may still be delayed at the recipient bank because of internal posting cycles, weekends, holidays, or manual compliance review. Confirm the business-day timeline before escalating."},

    {"doc_id": "kb_023", "title": "Wrong Transfer Details", "topic": "bank_transfer", "intent_hint": "beneficiary_not_allowed, transfer_not_received", "content": "If a transfer used incorrect account details, the transfer may fail, bounce back, or be delayed. Ask for the transfer reference and verify the recipient account, routing, and destination country before next action."},

    {"doc_id": "kb_024", "title": "Fraud Review on Transfer", "topic": "bank_transfer", "intent_hint": "outgoing_transfer_pending", "content": "Transfers may remain pending during fraud or compliance checks. These reviews often require no immediate customer action unless additional documents are requested. Escalate only after the normal review window has passed."},

    {"doc_id": "kb_025", "title": "Card Magnetic Stripe Issue", "topic": "cards", "intent_hint": "card_damage", "content": "Card failures at older terminals or ATMs may happen because of magnetic stripe damage, demagnetization, or terminal compatibility issues. If chip transactions also fail, replacement may be needed."},

    {"doc_id": "kb_026", "title": "ATM Dispensed Partial Cash", "topic": "disputes", "intent_hint": "cash_withdrawal_charge", "content": "If an ATM dispensed less cash than expected or no cash while the account was charged, collect the ATM location, date, amount, and receipt. These cases usually need a cash withdrawal dispute workflow."},

    {"doc_id": "kb_027", "title": "ATM Offline or Network Error", "topic": "cards", "intent_hint": "cash_withdrawal", "content": "Cash withdrawal can fail when the ATM is offline, out of cash, or experiencing network issues. Recommend trying another ATM before escalation if the card and balance appear normal."},

    {"doc_id": "kb_028", "title": "Merchant Terminal Issue", "topic": "card_payments", "intent_hint": "card_payment_fee_charged", "content": "A payment decline or unusual fee may be caused by merchant terminal configuration, offline authorization attempts, or duplicate processing. Ask whether the card worked elsewhere before assuming a card-side problem."},

    {"doc_id": "kb_029", "title": "International ATM Restrictions", "topic": "cards", "intent_hint": "cash_withdrawal", "content": "Cash withdrawals abroad may fail because of regional card controls, unsupported ATM networks, or local restrictions. Customers should verify international usage settings and any geographic limits in the app."},

    {"doc_id": "kb_030", "title": "Card Verification Pending", "topic": "security", "intent_hint": "card_not_received, cash_withdrawal", "content": "Some card actions remain restricted until identity checks or compliance verification are completed. If a customer reports unexpected restrictions, verify whether pending verification is blocking card usage or delivery actions."},

    {"doc_id": "kb_031", "title": "Duplicate Cash Withdrawal Charge", "topic": "disputes", "intent_hint": "cash_withdrawal_charge", "content": "If the customer sees duplicate ATM-related charges, check whether one entry is a temporary hold and whether both settled. Dispute only settled incorrect charges after reviewing transaction timestamps and ATM details."},

    {"doc_id": "kb_032", "title": "Transfer Scheduled on Weekend", "topic": "bank_transfer", "intent_hint": "outgoing_transfer_pending, transfer_not_received", "content": "Transfers initiated late on weekends or holidays may appear delayed even when submitted successfully. Business-day processing timelines should be explained before escalation."},

    {"doc_id": "kb_033", "title": "Card Used After Damage Report", "topic": "cards", "intent_hint": "card_damage", "content": "If a damaged card is still active, advise the customer to stop using it to avoid repeated payment or ATM failures. A replacement flow should be started after confirming card status and customer identity."},

    {"doc_id": "kb_034", "title": "Customer Needs ATM Receipt for Dispute", "topic": "disputes", "intent_hint": "cash_withdrawal_charge, cash_withdrawal", "content": "For ATM disputes, a receipt is helpful but not always mandatory. If no receipt is available, collect ATM location, date, approximate time, amount, and any on-screen error details."},

    {"doc_id": "kb_035", "title": "Courier Returned Card", "topic": "cards", "intent_hint": "card_not_received", "content": "A card may be returned by the courier if delivery failed, the address was incomplete, or nobody was available to receive it. Check whether the shipment status shows returned-to-sender before opening a replacement case."},

    {"doc_id": "kb_036", "title": "Cash Withdrawal Control Disabled", "topic": "security", "intent_hint": "cash_withdrawal", "content": "If cash withdrawal controls are disabled in the app, ATM transactions can be declined even when the card is valid and funded. Ask the customer to review cash withdrawal permissions, regional settings, and card lock status."},

    {"doc_id": "kb_037", "title": "Account Balance vs Available Balance", "topic": "cards", "intent_hint": "cash_withdrawal", "content": "A customer may see enough account balance but still fail an ATM withdrawal because available balance is lower due to pending holds, card authorizations, or unsettled transactions. Always verify available balance, not only ledger balance."},

    {"doc_id": "kb_038", "title": "Compliance Hold on Recipient", "topic": "bank_transfer", "intent_hint": "beneficiary_not_allowed", "content": "A beneficiary may be blocked by compliance screening because of destination-country restrictions, regulatory rules, or risk flags. These cases may require no retry until the compliance result is confirmed."},

    {"doc_id": "kb_039", "title": "Cash Deposit Receipt Missing", "topic": "cash_deposit", "intent_hint": "cash_deposit", "content": "If a cash deposit issue is reported without a receipt, gather ATM location, date, time, and approximate amount. Receipt absence does not always block investigation, but it can slow verification."},

    {"doc_id": "kb_040", "title": "Escalation Rules for Banking Support", "topic": "operations", "intent_hint": "cash_withdrawal, card_not_received, cash_deposit, outgoing_transfer_pending, transfer_not_received", "content": "Escalation should happen only after normal SLA windows, required verification, and basic troubleshooting are completed. First collect timeline, reference numbers, location details, and customer verification status."}
]


knowledge_docs = knowledge_docs + kb_extra_docs
kb_df = pd.DataFrame(knowledge_docs)
kb_df.head()

,doc_id,title,topic,intent_hint,content
0,kb_001,Card Arrived Damaged,cards,card_damage,"If a card arrives damaged, the customer should..."
1,kb_002,Cash Withdrawal Charge Policy,fees,cash_withdrawal_charge,"Cash withdrawal fees depend on ATM network, co..."
2,kb_003,Beneficiary Not Allowed,bank_transfer,beneficiary_not_allowed,A beneficiary may be blocked because of invali...
3,kb_004,Pending Bank Transfer,bank_transfer,outgoing_transfer_pending,"Pending transfers may be under fraud checks, c..."
4,kb_005,Cash Withdrawal Declined,cards,cash_withdrawal,Cash withdrawal failures can happen because of...


In [41]:
kb_chunks = []

def chunk_text(text, chunk_size=60, overlap=15):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(len(words), start + chunk_size)
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        
        start += chunk_size - overlap
        
    return chunks


for row in knowledge_docs:
    for i, ch in enumerate(chunk_text(row["content"], chunk_size=60, overlap=15)):
        kb_chunks.append({
            "chunk_id": f"{row['doc_id']}_chunk_{i}",
            "doc_id": row["doc_id"],
            "title": row["title"],
            "topic": row["topic"],
            "intent_hint": row["intent_hint"],
            "content": ch
        })

kb_chunks_df = pd.DataFrame(kb_chunks)
kb_chunks_df.head(1)

,chunk_id,doc_id,title,topic,intent_hint,content
0,kb_001_chunk_0,kb_001,Card Arrived Damaged,cards,card_damage,"If a card arrives damaged, the customer should..."


In [42]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

bm25 = BM25Okapi([x.split() for x in kb_chunks_df["content"].tolist()])
chunks_embeddings = embed_model.encode(
    kb_chunks_df["content"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
).astype("float32")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [43]:
index = faiss.IndexFlatIP(chunks_embeddings.shape[1])
index.add(chunks_embeddings)
print("FAISS index size:", index.ntotal)

FAISS index size: 40
